# Crack Spread Time-Only PTP Detailed Backtest

Loads the saved `*_best_model.pth` checkpoint and the exported validation split from
`crack_spread_tft_ptp_optuna.ipynb`, then runs a detailed OOS backtest without rebuilding
features from raw data. The workflow mirrors `mmtfv3_backtest_detailed.ipynb`:

- load artifacts and exported dataset
- restore the trained time-only PTP model
- run OOS inference on the saved validation split
- compute detailed backtest tables and plots
- export attribution and leakage-check artifacts


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'CTAFlow').exists() and (REPO_ROOT.parent / 'CTAFlow').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

RESULTS_PATH = REPO_ROOT / 'artifacts' / 'crack_spread_time_only_ptp'
MODEL_CANDIDATES = sorted(RESULTS_PATH.glob('*_best_model.pth'))
if not MODEL_CANDIDATES:
    raise FileNotFoundError(f'No *_best_model.pth checkpoint found under {RESULTS_PATH}')
MODEL_PATH = MODEL_CANDIDATES[-1]
STUDY_NAME = MODEL_PATH.name.replace('_best_model.pth', '')

EXPORT_DATASET_PATH = RESULTS_PATH / f'{STUDY_NAME}_optimized_datasets'
META_PATH = EXPORT_DATASET_PATH / f'{STUDY_NAME}_optimized_dataset_meta.json'
VAL_SAMPLES_PATH = EXPORT_DATASET_PATH / f'{STUDY_NAME}_val_samples.joblib'
BACKTEST_PATH = RESULTS_PATH / f'{STUDY_NAME}_detailed_backtest'
BACKTEST_PATH.mkdir(parents=True, exist_ok=True)

required_files = [MODEL_PATH, META_PATH, VAL_SAMPLES_PATH]
missing_required_files = [path for path in required_files if not path.exists()]
if missing_required_files:
    missing_text = '\n'.join(f'  - {path}' for path in missing_required_files)
    raise FileNotFoundError(f'Missing required artifacts:\n{missing_text}')

print(f'REPO_ROOT          : {REPO_ROOT}')
print(f'RESULTS_PATH       : {RESULTS_PATH}')
print(f'STUDY_NAME         : {STUDY_NAME}')
print(f'MODEL_PATH         : {MODEL_PATH}')
print(f'EXPORT_DATASET_PATH: {EXPORT_DATASET_PATH}')
print(f'BACKTEST_PATH      : {BACKTEST_PATH}')


In [ ]:
import json
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy import stats
from torch.utils.data import DataLoader

from CTAFlow.models.deep_learning.training import (
    TradingMode,
    run_detailed_backtest,
    save_detailed_backtest_artifacts,
)
from CTAFlow.models.deep_learning.multi_branch.tft.c_mmtft import PredictionToPosition
from notebooks.crack_spread_tft_ptp_support import (
    batch_to_device,
    compute_trading_metrics,
    load_time_only_export_split,
    load_time_only_ptp_checkpoint,
    time_ptp_collate_fn,
)

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


In [ ]:
with open(META_PATH) as f:
    export_meta = json.load(f)

model, ckpt = load_time_only_ptp_checkpoint(MODEL_PATH, device=device)
best = ckpt.get('best_params', export_meta.get('best_params', {}))
feature_cols = ckpt.get('feature_cols', export_meta.get('feature_cols', []))
val_samples, val_dataset = load_time_only_export_split(
    EXPORT_DATASET_PATH,
    STUDY_NAME,
    split='val',
    return_metadata=True,
)
BATCH_SIZE = int(best.get('batch_size', 64))
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=time_ptp_collate_fn,
    num_workers=0,
)

TC_COST = float(best.get('tc_cost', 0.0))
TRADE_THRESH = 0.0
ACTION_LABELS = list(ckpt.get('ptp_action_labels', PredictionToPosition.ACTION_LABELS))

print(f'Validation samples : {len(val_samples):,}')
print(f'Feature count      : {len(feature_cols)}')
print(f'Lookback steps     : {ckpt.get("lookback_steps")}')
print(f'Target column      : {ckpt.get("target_col")}')
print(f'TC_COST            : {TC_COST:.8f}')
print(f'Action labels      : {ACTION_LABELS}')


In [ ]:
all_positions = []
all_returns = []
all_logits = []
all_anchor_ts = []
all_target_end_ts = []

model.eval()
with torch.no_grad():
    for batch in val_loader:
        inputs, targets = batch_to_device(batch, device)
        position, logits, _ = model(**inputs)
        all_positions.append(position.detach().cpu().numpy().reshape(-1))
        all_returns.append(targets.detach().cpu().numpy().reshape(-1))
        all_logits.append(logits.detach().cpu().numpy())
        all_anchor_ts.extend(batch['_anchor_ts'])
        all_target_end_ts.extend(batch['_target_end_ts'])

bt_pos = np.concatenate(all_positions)
bt_ret = np.concatenate(all_returns)
bt_logits = np.concatenate(all_logits)
bt_probs = torch.softmax(torch.tensor(bt_logits, dtype=torch.float32), dim=-1).numpy()
bt_pred = bt_logits.argmax(axis=1)

bt = pd.DataFrame({
    'anchor_ts': pd.to_datetime(all_anchor_ts),
    'target_end_ts': pd.to_datetime(all_target_end_ts),
    'position': bt_pos,
    'fwd_return': bt_ret,
    'pred_class': bt_pred,
    'pred_label': [ACTION_LABELS[idx] for idx in bt_pred],
})
for idx, label in enumerate(ACTION_LABELS):
    bt[f'prob_{label}'] = bt_probs[:, idx]
    bt[f'logit_{label}'] = bt_logits[:, idx]

bt = bt.sort_values('anchor_ts', kind='stable').reset_index(drop=True)
bt['strategy_ret'] = bt['position'] * bt['fwd_return']
bt['active'] = bt['position'].abs() > TRADE_THRESH
bt['turnover'] = np.abs(np.diff(np.r_[0.0, bt['position'].to_numpy(dtype=float)]))
bt['tc_cost'] = bt['turnover'] * TC_COST
bt['strategy_ret_tc'] = bt['strategy_ret'] - bt['tc_cost']
bt['cum_strategy_ret'] = bt['strategy_ret'].cumsum()
bt['cum_strategy_ret_tc'] = bt['strategy_ret_tc'].cumsum()
bt['cum_forward_ret'] = bt['fwd_return'].cumsum()
bt['is_trade'] = bt['turnover'] > 0.0

bt_metrics = compute_trading_metrics(
    bt['position'].to_numpy(),
    bt['fwd_return'].to_numpy(),
    logits=bt_logits,
    outer_threshold=float(best.get('outer_threshold', 1.0)),
)
print(json.dumps(bt_metrics, indent=2, default=float))
print(f'OOS range: {bt["anchor_ts"].min()} -> {bt["anchor_ts"].max()}')
print(bt.head(3).to_string())


In [ ]:
BARS_PER_DAY = 26
BARS_PER_YEAR = 252 * BARS_PER_DAY
ANNUALIZATION = math.sqrt(BARS_PER_YEAR)

def _metrics(df, label, col='strategy_ret_tc'):
    sr = df[col].to_numpy(dtype=float)
    pos = df['position'].to_numpy(dtype=float)
    ret = df['fwd_return'].to_numpy(dtype=float)
    active = df['active'].to_numpy(dtype=bool)

    mean_sr = sr.mean() if len(sr) else 0.0
    std_sr = sr.std() + 1e-8
    neg = sr[sr < 0.0]
    downside = float(np.sqrt((neg ** 2).mean())) if len(neg) > 0 else 1e-8
    gp = sr[sr > 0.0].sum()
    gl = np.abs(sr[sr < 0.0]).sum() + 1e-9
    cum = np.cumsum(sr)
    run_max = np.maximum.accumulate(cum) if len(cum) else np.zeros((0,), dtype=float)
    mdd = float((run_max - cum).max()) if len(cum) else 0.0

    correct = ((pos > 0) & (ret > 0)) | ((pos < 0) & (ret < 0))
    dir_acc = float((correct & active).sum() / max(active.sum(), 1))
    win_rate = float((sr[active] > 0).mean() * 100) if active.sum() > 0 else 0.0
    sharpe_ann = (mean_sr / std_sr) * ANNUALIZATION if len(sr) else 0.0
    sortino_ann = (mean_sr / (downside + 1e-8)) * ANNUALIZATION if len(sr) else 0.0
    annual_ret = mean_sr * BARS_PER_YEAR
    calmar = annual_ret / (mdd + 1e-9) if mdd > 0 else 0.0
    n_trades = int(df['is_trade'].sum())
    avg_trade = float(df.loc[df['is_trade'], col].mean()) if n_trades > 0 else 0.0

    return {
        'Label': label,
        'N Samples': len(df),
        'Net PnL': round(float(sr.sum()), 6),
        'Ann. Return': round(float(annual_ret), 6),
        'Sharpe (ann)': round(float(sharpe_ann), 3),
        'Sortino (ann)': round(float(sortino_ann), 3),
        'Calmar': round(float(calmar), 3),
        'Win Rate (%)': round(float(win_rate), 2),
        'Dir Acc (%)': round(float(dir_acc * 100.0), 2),
        'Profit Factor': round(float(gp / gl), 4),
        'Max Drawdown': round(float(mdd), 6),
        '# Trades': n_trades,
        'Avg Trade PnL': round(float(avg_trade), 6),
        'Avg |Position|': round(float(np.abs(pos).mean()), 4),
        'Active Bars (%)': round(float(active.mean() * 100.0), 2),
    }

rows = [_metrics(bt, 'ALL — OOS (TC adj)')]
for label in ACTION_LABELS:
    df_label = bt[bt['pred_label'] == label]
    if len(df_label) > 0:
        rows.append(_metrics(df_label, f'{label} — OOS (TC adj)'))
df_summary = pd.DataFrame(rows).set_index('Label')
print(df_summary.to_string())

df_trades = bt.loc[bt['is_trade'], [
    'anchor_ts', 'target_end_ts', 'position', 'fwd_return', 'strategy_ret', 'tc_cost', 'strategy_ret_tc', 'pred_label'
]].copy()
print(f'Trades: {len(df_trades):,}')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(bt['anchor_ts'], bt['cum_strategy_ret'], label='gross')
axes[0, 0].plot(bt['anchor_ts'], bt['cum_strategy_ret_tc'], label='tc_adj')
axes[0, 0].plot(bt['anchor_ts'], bt['cum_forward_ret'], label='buy_hold', alpha=0.7)
axes[0, 0].legend()
axes[0, 0].set_title('Cumulative returns')

axes[0, 1].plot(bt['anchor_ts'], bt['position'], alpha=0.9)
axes[0, 1].set_title('Position path')

daily_pnl = bt.groupby(bt['anchor_ts'].dt.date)['strategy_ret_tc'].sum()
axes[1, 0].bar(pd.to_datetime(daily_pnl.index), daily_pnl.values, color=np.where(daily_pnl.values >= 0, '#2ecc71', '#e74c3c'))
axes[1, 0].set_title('Daily TC-adjusted PnL')

sns.countplot(data=bt, x='pred_label', order=ACTION_LABELS, ax=axes[1, 1], palette='viridis')
axes[1, 1].set_title('Predicted action distribution')
axes[1, 1].tick_params(axis='x', rotation=20)

for ax in axes.ravel():
    ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
detailed = run_detailed_backtest(
    model=model,
    loader=val_loader,
    device=device,
    batch_to_device_fn=batch_to_device,
    feature_names=feature_cols,
    gradient_keys=('features',),
    transaction_cost_bps=TC_COST * 10000.0,
    slippage_bps=0.0,
    mode=TradingMode.CONTINUOUS,
)
detail_paths = save_detailed_backtest_artifacts(
    detailed,
    BACKTEST_PATH,
    prefix=f'{STUDY_NAME}_validation_detail',
)

attn_df = detailed.attention_frame.copy()
attn_df = attn_df[attn_df['source'].astype(str).str.contains('attn_weights')].copy()
if not attn_df.empty:
    attn_df['lag_idx'] = attn_df['component'].astype(str).str.replace('lag_', '', regex=False).astype(int)
    mean_attn = attn_df.groupby('lag_idx')['weight'].mean().reset_index()
else:
    mean_attn = pd.DataFrame(columns=['lag_idx', 'weight'])

top_features = detailed.feature_summary.head(15).copy()
feature_wide = detailed.feature_importance_frame.pivot_table(
    index='anchor_ts', columns='feature', values='importance', aggfunc='mean'
).reset_index()
corr_rows = []
for feature in top_features['feature'].tolist():
    merged = bt[['anchor_ts', 'fwd_return']].merge(feature_wide[['anchor_ts', feature]], on='anchor_ts', how='inner').dropna()
    if len(merged) >= 3 and merged[feature].std() > 0:
        corr, pval = stats.pearsonr(merged[feature], merged['fwd_return'])
        corr_rows.append({'feature': feature, 'correlation': corr, 'p_value': pval})
leak_df = pd.DataFrame(corr_rows).sort_values('correlation', key=np.abs, ascending=False) if corr_rows else pd.DataFrame(columns=['feature', 'correlation', 'p_value'])

recent_attn_result = None
if not attn_df.empty:
    last_lags = sorted(attn_df['lag_idx'].unique())[-3:]
    recent_attn = (
        attn_df[attn_df['lag_idx'].isin(last_lags)]
        .groupby('anchor_ts')['weight']
        .mean()
        .reset_index()
    )
    merged_attn = bt[['anchor_ts', 'fwd_return']].merge(recent_attn, on='anchor_ts', how='inner').dropna()
    if len(merged_attn) >= 3 and merged_attn['weight'].std() > 0:
        corr_recent, p_recent = stats.pearsonr(merged_attn['weight'], merged_attn['fwd_return'])
        recent_attn_result = {'correlation': corr_recent, 'p_value': p_recent, 'lags': last_lags}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

feat_plot = top_features.sort_values('mean_importance', ascending=True)
axes[0, 0].barh(feat_plot['feature'], feat_plot['mean_importance'], color='#2ecc71', alpha=0.85)
axes[0, 0].set_title('Top gradient feature importances')
axes[0, 0].set_xlabel('Mean importance')

if not mean_attn.empty:
    axes[0, 1].bar(mean_attn['lag_idx'], mean_attn['weight'], color='steelblue', alpha=0.85)
axes[0, 1].set_title('Temporal attention by lookback lag')
axes[0, 1].set_xlabel('Lag index (0=oldest)')
axes[0, 1].set_ylabel('Mean attention weight')

if not leak_df.empty:
    leak_plot = leak_df.head(12).sort_values('correlation', key=np.abs, ascending=True)
    leak_colors = ['#e74c3c' if abs(v) > 0.05 and p < 0.01 else '#27ae60' for v, p in zip(leak_plot['correlation'], leak_plot['p_value'])]
    axes[1, 0].barh(leak_plot['feature'], leak_plot['correlation'], color=leak_colors, alpha=0.85)
axes[1, 0].axvline(0.0, color='gray', linestyle=':')
axes[1, 0].set_title('Leakage check: feature importance vs future return')
axes[1, 0].set_xlabel('Pearson correlation')

tracker_cols = [col for col in detailed.batch_tracker_frame.columns if col.startswith('ptp_')]
for col in tracker_cols:
    axes[1, 1].plot(detailed.batch_tracker_frame['batch_idx'], detailed.batch_tracker_frame[col], label=col)
axes[1, 1].set_title('Batch tracker series')
axes[1, 1].set_xlabel('Batch idx')
axes[1, 1].legend(loc='best')

for ax in axes.ravel():
    ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f'{STUDY_NAME}_validation_attribution.png', dpi=150, bbox_inches='tight')
plt.show()

if recent_attn_result is not None:
    print(
        f"Attention on recent lags {recent_attn_result['lags']} vs fwd return: "
        f"r={recent_attn_result['correlation']:.4f}  p={recent_attn_result['p_value']:.2e}"
    )
print(json.dumps(detailed.summary, indent=2, default=float))
display(detailed.feature_summary.head(20))
display(leak_df.head(20))


In [ ]:
bt_export_path = BACKTEST_PATH / f'{STUDY_NAME}_bt_full.csv'
summary_path = BACKTEST_PATH / f'{STUDY_NAME}_bt_summary.csv'
trades_path = BACKTEST_PATH / f'{STUDY_NAME}_bt_trades.csv'
leak_path = BACKTEST_PATH / f'{STUDY_NAME}_bt_leakage.csv'

bt.to_csv(bt_export_path, index=False)
df_summary.to_csv(summary_path)
df_trades.to_csv(trades_path, index=False)
leak_df.to_csv(leak_path, index=False)

print(f'Full backtest saved -> {bt_export_path}')
print(f'Summary saved       -> {summary_path}')
print(f'Trade table saved   -> {trades_path}')
print(f'Leakage saved       -> {leak_path}')
for key, path in detail_paths.items():
    print(f'{key:18s} -> {path}')
